In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('dataset_mansion.csv', index_col=0)

# Convert 'Consumption' to numeric, handling commas and ensuring proper data types
df['Consumption'] = pd.to_numeric(
    df['Consumption'].astype('string').str.replace(',', '.', regex=False),
    errors='raise'
)

# Convert 'MTime' to datetime, ensuring no missing or duplicate timestamps
df['MTime'] = pd.to_datetime(df['MTime'], errors='raise', utc=True)
assert df['MTime'].notna().all(), 'Check missing timestamps first.'
assert not df['MTime'].duplicated().any(), 'Check duplicate timestamps first.'
df = df.set_index('MTime').sort_index()

# Print the column names to verify the dataset structure
print("Displaying the column names of the dataset:")
print(df.columns.tolist())

# Check for missing values and handle them
df = df.asfreq('h')
print("--- Missing values BEFORE imputation ---")
missing_before = df.isna().sum()
print(missing_before[missing_before > 0])

# Impute missing values using time-based interpolation, followed by backfill and forward fill
df = df.interpolate(method='time')
df = df.bfill().ffill()

# Check for missing values after imputation
print("\n--- Missing values AFTER imputation ---")
print(df.isna().sum().sum()) # Should print 0

# Check for continuous timeline
expected = pd.date_range(df.index.min(), df.index.max(), freq='h')
assert df.index.equals(expected), 'Timeline is still not continuous!'

# Preserve the original dataframe
df_features = df.copy()

local_time = df_features.index.tz_convert('Europe/Helsinki')

# Target timestamp's hour: 0–23
hour = local_time.hour

df_features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Weekday = 0, weekend = 1
df_features['is_weekend'] = (
    local_time.dayofweek >= 5
).astype(int)

# Month: convert January–December to 0–11 before encoding
month = local_time.month - 1

df_features['month_sin'] = np.sin(2 * np.pi * month / 12)
df_features['month_cos'] = np.cos(2 * np.pi * month / 12)

# Initial feature-removal choices
columns_to_drop = [
    'Latitude',
    'Longitude',
    'Place',
    'PresentWeather(rank)',
    'HighestTemperature(degC)',
    'LowestTemperature(degC)',
    'MaximumWindSpeed(m/s)',
    'MinimumWindSpeed(m/s)',
    'WindDirection(deg)',
    'MaximumPrecipitationIntensity(mm/h)'
]
df_features = df_features.drop(columns=columns_to_drop)

# --- 1. COVID-19 Era ---
# "COVID-19 epidemic began to significantly affect everyday life in Finland in March 2020"
# This likely shifted the family to remote work, altering baseline daytime consumption.
df_features['is_covid'] = (local_time >= '2020-03-01').astype(int)

# --- 2. Holiday Periods ---
# "Holidays week 9 and 42, and of course christmas holiday"
# "holidays start from the second week of June and end the first of August"
week_of_year = local_time.isocalendar().week
month = local_time.month
day = local_time.day

# Define the complex summer holiday logic (roughly June 8th through August 7th)
is_summer_holiday = ((month == 6) & (day >= 8)) | (month == 7) | ((month == 8) & (day <= 7))

# Define fixed holiday weeks and Christmas (Dec 24-26)
is_week_9 = (week_of_year == 9)
is_week_42 = (week_of_year == 42)
is_christmas = ((month == 12) & (day >= 24) & (day <= 26))

# Combine all holidays into one flag
df_features['is_holiday'] = (is_summer_holiday | is_week_9 | is_week_42 | is_christmas).astype(int)

# --- 3. Working Hours ---
# "Working time 8-16"
# During these hours, the family is away IF it is not a weekend and not a holiday.
df_features['is_working_hour'] = (
    (hour >= 8) & 
    (hour < 16) & 
    (df_features['is_weekend'] == 0) & 
    (df_features['is_holiday'] == 0)
).astype(int)

# --- 4. Fireplace / Evening Routine ---
# "Fire place... after 8 p.m." and "dishing machine, 1 washing machine, used normally after work"
# A flag for the active evening hours (e.g., 17:00 to 23:00) when consumption spikes.
df_features['is_evening_routine'] = ((hour >= 17) & (hour <= 23)).astype(int)

print(df_features.columns.tolist())

from sklearn.preprocessing import StandardScaler

# DATASET SPLITTING (Run this FIRST)
train_df = df_features.loc['2016-01-01':'2020-09-30'].copy()
test_df = df_features.loc['2020-10-01':'2021-09-30'].copy()
eval1_df = df_features.loc['2021-10-01':'2021-12-31'].copy()
eval2_df = df_features.loc['2022-01-01':'2022-02-28'].copy()
eval3_df = df_features.loc['2022-03-01':'2022-05-31'].copy()

# DATA SCALING (Using StandardScaler)
continuous_features = [
    'AirTemperature(degC)', 
    'RelativeHumidity(%)', 
    'WindSpeed(m/s)', 
    'PrecipitationAmount(mm)', 
    'AirPressure(hPa)'
]

# Initialize the StandardScaler
scaler = StandardScaler()

# FIT and transform ONLY on the training data
train_df[continuous_features] = scaler.fit_transform(train_df[continuous_features])

# TRANSFORM only (do not fit) the test and eval sets
test_df[continuous_features] = scaler.transform(test_df[continuous_features])
eval1_df[continuous_features] = scaler.transform(eval1_df[continuous_features])
eval2_df[continuous_features] = scaler.transform(eval2_df[continuous_features])
eval3_df[continuous_features] = scaler.transform(eval3_df[continuous_features])

print(f"Training set: {len(train_df)} hours")
print("Standard scaling applied successfully.")


Displaying the column names of the dataset:
['Consumption', 'Latitude', 'Longitude', 'Place', 'AirTemperature(degC)', 'HighestTemperature(degC)', 'LowestTemperature(degC)', 'RelativeHumidity(%)', 'WindSpeed(m/s)', 'MaximumWindSpeed(m/s)', 'MinimumWindSpeed(m/s)', 'WindDirection(deg)', 'PrecipitationAmount(mm)', 'MaximumPrecipitationIntensity(mm/h)', 'AirPressure(hPa)', 'PresentWeather(rank)']
--- Missing values BEFORE imputation ---
Latitude                                 191
Longitude                                191
Place                                    191
AirTemperature(degC)                     208
HighestTemperature(degC)                 203
LowestTemperature(degC)                  203
RelativeHumidity(%)                      289
WindSpeed(m/s)                           618
MaximumWindSpeed(m/s)                    741
MinimumWindSpeed(m/s)                    741
WindDirection(deg)                       501
PrecipitationAmount(mm)                  303
MaximumPrecipitationInt

/var/folders/0t/k860jmxx4d5__9h3p08nh77m0000gn/T/ipykernel_56634/763769958.py:30: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.interpolate(method='time')


Training set: 41640 hours
Standard scaling applied successfully.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Define the TimeSeriesDataset class
class TimeSeriesDataset(Dataset):
    def __init__(self, df, target_col_name, seq_len=48, pred_len=1):
        self.data = df.values.astype(np.float32)
        self.target_idx = df.columns.get_loc(target_col_name)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, index):
        # X: Past 'seq_len' hours of all features
        x = self.data[index : index + self.seq_len]
        # Y: The next 1 hour of Consumption[cite: 6]
        y = self.data[index + self.seq_len : index + self.seq_len + self.pred_len, self.target_idx]
        return torch.tensor(x), torch.tensor(y)

SEQ_LEN = 48  # Look back 48 hours to predict the next hour
BATCH_SIZE = 64

# Create Datasets using your split dataframes
train_dataset = TimeSeriesDataset(train_df, 'Consumption', SEQ_LEN)
test_dataset = TimeSeriesDataset(test_df, 'Consumption', SEQ_LEN)
eval1_dataset = TimeSeriesDataset(eval1_df, 'Consumption', SEQ_LEN)
eval2_dataset = TimeSeriesDataset(eval2_df, 'Consumption', SEQ_LEN)
eval3_dataset = TimeSeriesDataset(eval3_df, 'Consumption', SEQ_LEN)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval1_loader = DataLoader(eval1_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval2_loader = DataLoader(eval2_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval3_loader = DataLoader(eval3_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [3]:
# Define the SimpleTCN model

class SimpleTCN(nn.Module):
    def __init__(self, num_features, num_channels, kernel_size=3):
        super(SimpleTCN, self).__init__()
        layers = []
        in_channels = num_features
        
        # Build 1D Convolutional layers with increasing dilations (1, 2, 4, 8)
        for out_channels in num_channels:
            dilation = len(layers) // 2 + 1 
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size, padding=(kernel_size-1)*dilation, dilation=dilation),
                nn.ReLU(),
                nn.Dropout(0.2)
            ]
            in_channels = out_channels
            
        self.tcn = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], 1) # Output next hour prediction

    def forward(self, x):
        # PyTorch Conv1d expects shape: [Batch, Features, Sequence_Length]
        x = x.permute(0, 2, 1) 
        
        y = self.tcn(x)
        # Take the output of the last time step
        y = y[:, :, -1] 
        return self.linear(y)

# Initialize Model
NUM_FEATURES = train_df.shape[1]
model = SimpleTCN(num_features=NUM_FEATURES, num_channels=[32, 32, 64, 64])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.MSELoss() # Assignment requires appropriate loss function[cite: 6]
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- TRAINING LOOP ---
epochs = 10
print("Starting Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss/len(train_loader):.4f}")

Starting Training...
Epoch 1/10 | Train MSE Loss: 1.1804
Epoch 2/10 | Train MSE Loss: 0.9142
Epoch 3/10 | Train MSE Loss: 0.8872
Epoch 4/10 | Train MSE Loss: 0.8706
Epoch 5/10 | Train MSE Loss: 0.8568
Epoch 6/10 | Train MSE Loss: 0.8489
Epoch 7/10 | Train MSE Loss: 0.8347
Epoch 8/10 | Train MSE Loss: 0.8214
Epoch 9/10 | Train MSE Loss: 0.8233
Epoch 10/10 | Train MSE Loss: 0.8155


In [4]:
import math

# Define a function to evaluate the model on a given DataLoader
def evaluate_model(loader, dataset_name):
    model.eval()
    mse_loss = 0.0
    mae_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            predictions = model(x_batch)
            
            mse_loss += nn.MSELoss(reduction='sum')(predictions, y_batch).item()
            mae_loss += nn.L1Loss(reduction='sum')(predictions, y_batch).item()
            total_samples += y_batch.size(0)
            
    mse = mse_loss / total_samples
    rmse = math.sqrt(mse)
    mae = mae_loss / total_samples
    return rmse, mae

# Gather results
results = {
    "Dataset": ["Training", "Test (2020-2021)", "Eval 1 (Late 2021)", "Eval 2 (Early 2022)", "Eval 3 (Spring 2022)"],
    "RMSE (kWh Error)": [
        evaluate_model(train_loader, "Train")[0],
        evaluate_model(test_loader, "Test")[0],
        evaluate_model(eval1_loader, "Eval 1")[0],
        evaluate_model(eval2_loader, "Eval 2")[0],
        evaluate_model(eval3_loader, "Eval 3")[0]
    ],
    "MAE (kWh Error)": [
        evaluate_model(train_loader, "Train")[1],
        evaluate_model(test_loader, "Test")[1],
        evaluate_model(eval1_loader, "Eval 1")[1],
        evaluate_model(eval2_loader, "Eval 2")[1],
        evaluate_model(eval3_loader, "Eval 3")[1]
    ]
}

# Display as a clean table
results_df = pd.DataFrame(results)
print("\n--- Final Model Accuracy Table ---")
print(results_df.round(3).to_string(index=False))


--- Final Model Accuracy Table ---
             Dataset  RMSE (kWh Error)  MAE (kWh Error)
            Training             0.869            0.643
    Test (2020-2021)             0.892            0.662
  Eval 1 (Late 2021)             0.975            0.749
 Eval 2 (Early 2022)             1.071            0.823
Eval 3 (Spring 2022)             0.846            0.668
